# **Qdrant VectorDB**

## **Installation Qdrant**

In [1]:
# ! pip install qdrant-client

In [2]:
! pip show qdrant-client

Name: qdrant-client
Version: 1.17.1
Summary: Client library for the Qdrant vector search engine
Home-page: https://github.com/qdrant/qdrant-client
Author: Andrey Vasnetsov
Author-email: andrey@qdrant.tech
License: Apache-2.0
Location: /Users/kanavbansal/Developer/.env_jupyter/lib/python3.13/site-packages
Requires: grpcio, httpx, numpy, portalocker, protobuf, pydantic, urllib3
Required-by: 


## **Initialize the client**

In [3]:
from qdrant_client import QdrantClient

client = QdrantClient(path="./qdrant_store")

# client.close()

In [4]:
client.info()

VersionInfo(title='qdrant - vector search engine', version='1.17.1', commit=None)

In [5]:
client.get_collections()

CollectionsResponse(collections=[])

## **Create a collection**

In [6]:
from qdrant_client.models import VectorParams, Distance, HnswConfigDiff, StrictModeConfig

client.create_collection(
    collection_name="clothes",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE,
        hnsw_config=HnswConfigDiff(
            m=0,  # Bulk load fast: m=0 (build links after ingest).
            ef_construct=50,  # Build quality: used after we set m>0
            full_scan_threshold=10,  # force HNSW instead of full scan
        ),
    ),
)

True

In [7]:
# Verify collection settings
collection_info = client.get_collection(collection_name="clothes")

print(f"Vector size: {collection_info.config.params.vectors.size}")
print(f"Distance metric: {collection_info.config.params.vectors.distance}")
print(f"HNSW: {collection_info.config.hnsw_config}")

Vector size: 384
Distance metric: Cosine
HNSW: m=16 ef_construct=100 full_scan_threshold=10000 max_indexing_threads=0 on_disk=None payload_m=None inline_storage=None


### **HNSW Parameters**

1. Graph Connectivity: The **m** parameter controls the maximum number of connections per node in the graph.
2. Build Thoroughness: The **ef_construct** parameter controls how many candidates are checked while inserting a new vector.
3. Search Thoroughness: The **hnsw_ef** parameter determines the number of candidates evaluated during a search query.

**Important Considerations:**
- High-speed retrieval: lower m and hnsw_ef; set ef_construct just high enough for acceptable recall.
- Maximum recall: raise m, hnsw_ef, and ef_construct and accept slower queries and builds.
- Tight RAM: reduce m; keep ef_construct high enough to avoid poor links.

```python
from qdrant_client.models import HnswConfig

# Example m values
fast_config = HnswConfig(m=8, ef_construct=100, full_scan_threshold=10000)      # Lower recall, less memory, faster build
balanced_config = HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000) # Default - good balance
accurate_config = HnswConfig(m=32, ef_construct=100, full_scan_threshold=10000) # Better recall, more memory, slower build
```
**[Click Here](https://qdrant.tech/course/essentials/day-2/what-is-hnsw/) for HNSW Reference.**

In [13]:
from qdrant_client.models import HnswConfigDiff

client.update_collection(
    collection_name="clothes",
    hnsw_config=HnswConfigDiff(
        m=16,
        ef_construct=200
    )
)

False

In [ ]:
collection_info = client.get_collection(collection_name="clothes")

collection_info.config.hnsw_config

## **Add vectors**

### **Step a: Load the data**

In [7]:
examples = [
    ("Sleeveless maxi dress with a V-neckline and wrap front."
     "Comes with a tie belt at the waist.", 29.95),
    ("Slim-fit jeans in washed stretch denim with a button fly "
     "and tapered legs.", 39.99),
    ("Double-breasted blazer in textured-weave fabric with peak "
     "lapels and front flap pockets.", 59.50),
    ("Lightweight bomber jacket with ribbed cuffs, a baseball collar, "
     "and zip front closure.", 45.00),
    ("Chunky knit sweater with dropped shoulders and ribbed trim around "
     "the neck, cuffs, and hem.", 25.99),
    ("Tailored trousers in a smooth woven fabric with a concealed "
     "hook-and-eye closure and welt back pockets.", 34.99),
    ("Classic trench coat with adjustable belt, storm flap, and button "
     "front closure.", 79.90),
    ("High-rise pencil skirt in a stretch fabric with a hidden rear zip "
     "and back slit.", 22.99),
    ("Athletic-fit polo shirt in moisture-wicking fabric with a ribbed "
     "collar and two-button placket.", 19.95),
    ("Soft flannel pajama set with long sleeves, matching pants, and a "
     "comfortable elastic waistband.", 32.00),
    ("Quilted puffer jacket with a detachable hood and zippered side "
     "pockets.", 48.99),
    ("Cropped denim jacket with distressed details and button-flap chest "
     "pockets.", 36.50),
    ("Fitted bodysuit with a scoop neckline and snap-button closure at "
     "the bottom.", 15.99),
    ("Lightly padded parka with a faux fur-lined hood, drawstring waist, "
     "and snap front pockets.", 69.95),
    ("Mesh panel sports leggings with a high waist and reflective details "
     "for nighttime visibility.", 27.99),
    ("Button-up cardigan in a soft knit with long sleeves and ribbed "
     "trim.", 24.50),
    ("Leather moto jacket with zippered cuffs, a notched collar, and "
     "asymmetrical zip closure.", 95.00),
    ("Velvet slip dress with a lace trim neckline and adjustable "
     "spaghetti straps.", 31.99),
    ("Cargo shorts with multiple pockets and a durable belt loop "
     "waistband.", 22.95),
    ("Wide-leg palazzo pants with a high-rise fit and side zip "
     "closure.", 38.99),
    ("Graphic print tee featuring an original artwork design and classic "
     "crew neck.", 14.99),
    ("Boho-style maxi skirt with an elastic waistband and tiered ruffle "
     "detailing.", 33.50),
    ("Men's linen shirt with a Mandarin collar and buttoned chest "
     "pocket.", 29.95),
    ("Cable knit beanie with a fold-over cuff and soft fleece "
     "lining.", 12.99),
    ("Sequin cocktail dress with a plunging V-neck and bodycon "
     "fit.", 49.99),
]

In [8]:
examples[0]

('Sleeveless maxi dress with a V-neckline and wrap front.Comes with a tie belt at the waist.',
 29.95)

### **Step b: Load the Embd Model**

In [9]:
from sentence_transformers import SentenceTransformer, util

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

sbert_model.encode("hi")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


array([-9.04762074e-02,  4.04396094e-02,  2.39056516e-02,  5.89480214e-02,
       -2.28823330e-02, -4.72200625e-02,  4.50475886e-02,  1.57863274e-02,
       -4.81995158e-02, -3.77940834e-02, -1.90776009e-02,  2.13088449e-02,
       -4.68305871e-03, -4.33081873e-02,  5.99148050e-02,  5.91033883e-02,
       -2.80367322e-02, -5.92183433e-02, -1.24403127e-01, -3.56000364e-02,
       -6.08061440e-03,  3.24290134e-02, -3.78007106e-02,  2.47110073e-02,
       -4.27243076e-02, -4.24539112e-02,  4.59356345e-02,  9.86255407e-02,
       -4.99980189e-02, -3.52358297e-02,  7.08397478e-02,  3.31632271e-02,
        2.65883207e-02,  1.73190594e-04,  3.88160697e-03,  3.04672904e-02,
       -7.82026276e-02, -1.20379575e-01,  1.80415213e-02,  2.28290688e-02,
       -1.77500898e-03, -2.34498568e-02,  3.05810850e-03,  2.43557282e-02,
        4.41539735e-02, -4.01097387e-02,  2.01923605e-02,  1.08881602e-02,
        2.87315324e-02,  1.23676872e-02, -9.13190693e-02, -6.81244507e-02,
        6.19148184e-03, -

### **Step c: Inject the Vectors in the Qdrant DB**

In [12]:
import uuid
from qdrant_client.models import PointStruct

client.upsert(
    collection_name="clothes",
    points=[
        PointStruct(
            id=uuid.uuid4(),
            vector=sbert_model.encode(description),
            payload={"description": description, "price": price},
        )
        for description, price in examples
    ]
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

## **Run a Query**

In [14]:
search_results = client.query_points(
    collection_name="clothes",
    query=sbert_model.encode("for cold weather"),
    with_payload=True,
    limit=3,
).points

for result in search_results:
    print(result)
    print()

id='fc25ab65-e197-4ee2-9a45-2d5f1c703e6b' version=0 score=0.2653765188490913 payload={'description': 'Quilted puffer jacket with a detachable hood and zippered side pockets.', 'price': 48.99} vector=None shard_key=None order_value=None

id='53ec1554-a570-4e65-8ee2-bc6d000cf96d' version=0 score=0.25686171229084354 payload={'description': 'Classic trench coat with adjustable belt, storm flap, and button front closure.', 'price': 79.9} vector=None shard_key=None order_value=None

id='5dc88b9c-27f9-465e-ad3f-0ac373cc040f' version=0 score=0.255167471761402 payload={'description': 'Lightweight bomber jacket with ribbed cuffs, a baseball collar, and zip front closure.', 'price': 45.0} vector=None shard_key=None order_value=None



In [15]:
search_results = client.query_points(
    collection_name="clothes",
    query=sbert_model.encode("for cold weather above $40"),
    with_payload=True,
    limit=3,
).points

for result in search_results:
    print(result)
    print()

id='a657d2dd-dd4b-4f22-a9ab-a1e994cd319f' version=0 score=0.23303535444051968 payload={'description': 'Mesh panel sports leggings with a high waist and reflective details for nighttime visibility.', 'price': 27.99} vector=None shard_key=None order_value=None

id='53ec1554-a570-4e65-8ee2-bc6d000cf96d' version=0 score=0.219211611975558 payload={'description': 'Classic trench coat with adjustable belt, storm flap, and button front closure.', 'price': 79.9} vector=None shard_key=None order_value=None

id='fc25ab65-e197-4ee2-9a45-2d5f1c703e6b' version=0 score=0.1872143582095778 payload={'description': 'Quilted puffer jacket with a detachable hood and zippered side pockets.', 'price': 48.99} vector=None shard_key=None order_value=None



## **Run a Query with Metadata Filter**

A key feature of Qdrant is the **effective combination of vector and traditional indexes**. It is essential to have this because for vector search to work effectively with filters, having a vector index only is not enough. In simpler terms, a **vector index speeds up vector search**, and **payload indexes speed up filtering**.


**Important:**
- It's highly recommended to create all payload indices immediately after collection creation.
- Payload index may occupy some additional memory, so it is recommended to only use the index for those fields that are used in filtering conditions.
- If you need to filter by many fields and the memory limits do not allow for indexing all of them, it is recommended to choose the field that limits the search result the most. As a rule, the more different values a payload value has, the more efficiently the index will be used.

In [16]:
from qdrant_client.models import FloatIndexParams, FloatIndexType

client.create_payload_index(
    collection_name="clothes",
    field_name="price",                # name_of_the_field_to_index
    field_schema=FloatIndexParams(
        type=FloatIndexType.FLOAT,
        on_disk=True
    )
)

# IntegerIndexParams -> IntegerIndexType.INTEGER
# KeywordIndexParams -> KeywordIndexType.KEYWORD

/var/folders/lk/kxbf6qq17z3f79hl8dyn8xr40000gn/T/ipykernel_29188/316274923.py:3: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

**Available field types are:**
- **keyword** - for keyword payload, affects Match filtering conditions.
- **integer** - for integer payload, affects Match and Range filtering conditions.
- **float** - for float payload, affects Range filtering conditions.
- **bool** - for bool payload, affects Match filtering conditions (available as of v1.4.0).
- **geo** - for geo payload, affects Geo Bounding Box and Geo Radius filtering conditions.
- **datetime** - for datetime payload, affects Range filtering conditions (available as of v1.8.0).
- **text** - a special kind of index, available for keyword / string payloads, affects Full Text search filtering conditions. Read more about text index configuration
- **uuid** - a special type of index, similar to keyword, but optimized for UUID values. Affects Match filtering conditions. (available as of v1.11.0)

In [19]:
from qdrant_client.models import Filter, FieldCondition, Range

search_results = client.query_points(
    collection_name="clothes",
    query=sbert_model.encode("for cold weather above $40"),
    query_filter=Filter(
        must=[
            FieldCondition(
                key="price",
                range=Range(gte=40),
            ),
        ]
    ),
    with_payload=True,
    limit=3,
).points

for result in search_results:
    print(result)
    print()

id='53ec1554-a570-4e65-8ee2-bc6d000cf96d' version=0 score=0.219211611975558 payload={'description': 'Classic trench coat with adjustable belt, storm flap, and button front closure.', 'price': 79.9} vector=None shard_key=None order_value=None

id='fc25ab65-e197-4ee2-9a45-2d5f1c703e6b' version=0 score=0.1872143582095778 payload={'description': 'Quilted puffer jacket with a detachable hood and zippered side pockets.', 'price': 48.99} vector=None shard_key=None order_value=None

id='5dc88b9c-27f9-465e-ad3f-0ac373cc040f' version=0 score=0.16952052769354997 payload={'description': 'Lightweight bomber jacket with ribbed cuffs, a baseball collar, and zip front closure.', 'price': 45.0} vector=None shard_key=None order_value=None

